## Random Recommender (Baseline A)

This notebook implements a random baseline recommender system. The model recommends restaurants uniformly at random from the set of restaurants that a user has not previously interacted with in the training data.

The purpose of this baseline is to establish a lower bound for performance to compare other recommenders against.

In [52]:
import pandas as pd
import numpy as np
import math

# loading data
train_df = pd.read_csv("train_reviews_santa_barbara.csv")
test_df = pd.read_csv("test_reviews_santa_barbara.csv")
restaurants_df = pd.read_csv("restaurants_santa_barbara.csv")

# preprocessing
# set of all restaurants
all_restaurants = set(restaurants_df["business_id"].unique())

# dictionary mapping each user to set of restaurants they reviewed in training data
train_restaurants = {}
train_restaurants_groups = train_df.groupby("user_id")["business_id"]
for user, items in train_restaurants_groups:
    train_restaurants[user] = set(items)

# dictionary mapping each user to singular restaurant in testing data
test_restaurants = {}
for _, row in test_df.iterrows():
    test_restaurants[row["user_id"]] = row["business_id"]

recommendations = {}

rng = np.random.default_rng(42)
for user in test_restaurants:
    # get list of restaurants the user has not seen before
    seen_restaurants = train_restaurants.get(user, set())
    unseen_restaurants = list(all_restaurants - seen_restaurants)

    # randomly shuffle unseen restaurants
    rng.shuffle(unseen_restaurants)
    # recommend top 30 (need 30 to test Hit/NDCG @ 10, 20, 30
    recommendations[user] = unseen_restaurants[:30]

## Evaluation Metrics: Hit@k and NDCG@k

**Hit@K**: Measures whether the true test restaurant appears in the top-K recommendations

**NDCG@K**: Measures the ranking quality by assigning higher scores when the true restaurant is ranked higher in the recommendation list

In [54]:
def hit_at_k(k):
    """
    Computes hit@k for the recommender

    For each user, this metric checks whether the true/held-out test restaurant
    appears in the top-K recommended restaurants. The hit rate for each restaurant
    is either 0 or 1 (0 if it does not appear in top-K recommendations and 1 if it
    does appear in top-K). This returns the average hit rate across all users

    Parameters:
    k: int
        The cutoff rank (10, 20, 30) for evaluating recommendations

    Returns:
    float
        The average hit@k score across all users, the fraction of users whose true test
        restaurant appears in the top-K recommendations
    """
    hits = []
    for user, true_restaurant in test_restaurants.items():
        recs = recommendations[user]
        if true_restaurant in recs[0:k]:
            hits.append(1)
        else:
            hits.append(0)
    score = sum(hits) / len(hits)
    return score

def ndcg_at_k(k):
    """
    Computes Normalized Discounted Cumulative Gain (NDCG@K).

    For each user, this metric measures how highly the true (held-out) test
    restaurant is ranked in the top-K recommendations. Higher ranks receive
    higher scores, with a logarithmic discount applied to lower ranks.

    Since each user has only one relevant item, the ideal DCG (IDCG) is rank 1
    NDCG = DCG / IDCG, then average across all users

    Parameters:
    k: int
        The cutoff rank (10, 20, 30) for evaluating recommendations

    Returns:
    float
        The average NDCG@K score across all users
    """
    ndcgs = []

    for user, true_restaurant in test_restaurants.items():
        top_k = recommendations[user][:k]

        if true_restaurant in top_k:
            rating = test_df.loc[test_df["user_id"] == user, "stars"].iloc[0]
            rank = top_k.index(true_restaurant) + 1
            dcg = rating / math.log2(rank + 1)
            # IDCG is when restaurant is at rank 1
            idcg = rating / math.log2(1 + 1)
            ndcg = dcg / idcg
        else:
            ndcg = 0.0

        ndcgs.append(ndcg)

    return sum(ndcgs) / len(ndcgs)


## Reporting final evaluation metrics

Evaluating Hit@10, Hit@20, Hit@30 and NDCG@10, NDCG@20, NDCG@30

In [55]:
print("Random Recommender: Evaluation Metrics")

for k in [10, 20, 30]:
    print(f"Hit@{k}:", hit_at_k(k), f"\t\tNDCG@{k}:", ndcg_at_k(k))

Random Recommender: Evaluation Metrics
Hit@10: 0.011455946677775464 		NDCG@10: 0.005306903799796958
Hit@20: 0.023328473234742762 		NDCG@20: 0.008263653070472679
Hit@30: 0.03790876900645699 		NDCG@30: 0.011366661155617106
